In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

In [0]:
bronze_company_df = spark.table("raj.bronze.company_info")

bronze_company_df.printSchema()

In [0]:
for col_name in bronze_company_df.columns:
    null_cnt = bronze_company_df.filter(col(col_name).isNull()).count()
    print(f"Column {col_name} has {null_cnt} null values")

total = bronze_company_df.count()
distinct = bronze_company_df.select(col("ticker")).distinct().count()
print("DUPLICATES:",total-distinct)

In [0]:
silver_company_df = bronze_company_df.withColumn("sector",upper(col("sector"))).withColumn("industry",upper(col("industry")))\
    .withColumn("country",upper(col("country"))).withColumn("company_name",upper(col("company_name"))).withColumn("silver_timestamp",lit(datetime.now()))\
    .withColumn("isValid",when(
        (col("ticker").isNotNull()) &
        (col("company_name").isNotNull()) &
        (col("sector").isNotNull()) &
        (col("industry").isNotNull()),True).otherwise(False)
    ).dropDuplicates(["ticker"]).orderBy("ticker")

display(silver_company_df)



In [0]:
silver_company_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("raj.silver.company_info")